# Predict expression modulation targets

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you should be able to:

1. Explain why knockouts alone are often not enough, and what over-expression and down-regulation targets are.
2. Run FSEOF and identify reactions whose flux rises (or falls) as product formation is enforced.
3. Run differential flux variability analysis and read its normalised flux-range comparison.
4. Critically filter predicted targets (exchange, transport and pseudo-reactions are not genetic targets).

**Estimated time:** 60 min (including the exercise).

**Prerequisites:** FVA and production envelopes (notebook 04), knockout design (notebook 09).

### Background

Knockouts can only switch reactions *off*. Many successful cell factories also need enzymes that are **over-expressed** (to pull more flux through a bottleneck) or **down-regulated** (to reduce, but not remove, a competing flux that is still needed for growth). Cameo ([Cardoso et al., 2018](https://doi.org/10.1021/acssynbio.7b00423)) provides two constraint-based methods to suggest such modulation targets:

* **FSEOF**, flux scanning based on enforced objective flux ([Choi et al., 2010](https://doi.org/10.1128/aem.00115-10)). The product flux is forced to increasing values between its minimum and its theoretical maximum, and at each step the model is solved for maximal growth. Reactions whose flux magnitude grows together with the enforced product flux are candidate **amplification** (over-expression) targets; reactions whose flux shrinks are candidate down-regulation targets.
* **Differential FVA** compares flux *ranges* rather than single fluxes. Flux variability analysis is run for a reference state (here the wild type growing at its maximum rate) and for a grid of points along the surface of the production envelope (from high growth with low production to low growth with high production). A reaction whose flux range moves up, moves down, reverses direction or collapses to zero between reference and production states is a candidate for over-expression, down-regulation, flux reversal or knockout.

Both methods are *predictions from stoichiometry only*: they know nothing about enzyme kinetics, regulation or protein cost. Treat their output as a ranked list of hypotheses to test in the lab.

In [ ]:
from cobra.io import read_sbml_model

Load the *E. coli* core model. We look at succinate, a platform chemical that *E. coli* secretes during anaerobic fermentation.

In [ ]:
model = read_sbml_model('data/e_coli_core.xml.gz')

## Succinate production

The production envelope shows the maximum (and minimum) succinate secretion for each growth rate. The theoretical maximum succinate flux is reached at zero growth, and the trade-off between growth and production defines the space FSEOF will scan.

In [ ]:
import cameo_compat  # noqa: F401 (patches cameo for modern Python/cobra)
from cameo.flux_analysis.analysis import phenotypic_phase_plane
from cameo.visualization.plotting.with_plotly import PlotlyPlotter
plotter = PlotlyPlotter()
production_envelope = phenotypic_phase_plane(model, 
                                             variables=[model.reactions.BIOMASS_Ecoli_core_w_GAM],
                                             objective=model.metabolites.succ_e)
production_envelope.plot(plotter, height=400)

## Flux Scanning based on Enforced Objective Flux

`FSEOF(model)` prepares the scan; `run(target=...)` enforces succinate secretion at 10 evenly spaced levels (columns 1 to 10) between the flux at maximal growth and the theoretical maximum, and maximises growth at each level.

In [ ]:
from cameo.strain_design.deterministic.flux_variability_based import FSEOF

In [ ]:
fseof = FSEOF(model)

In [ ]:
result = fseof.run(target=model.reactions.EX_succ_e)

In [ ]:
result.data_frame

The result table has one row per reaction whose flux changes consistently during the scan, and one column per enforced succinate level. Reactions whose absolute flux increases from column 1 to column 10 (for example the fumarate reductase FRD7 and the glyoxylate shunt enzymes ICL and MALS) are candidate over-expression targets.

Plotting the table (reactions as lines, enforced levels on the x-axis) makes rising and falling fluxes easy to spot.

In [ ]:
from matplotlib import pyplot as plt
result.data_frame.T.plot()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

## Differential flux variability analysis

Compares flux ranges of a reference model to a set of models that have been parameterized to lie on a grid of evenly spaced points in the n-dimensional production envelope (n being the number of reaction bounds to be varied).

Differential FVA can suggest all kinds of targets at once: over-expression (range shifts to higher absolute fluxes), down-regulation (range shifts towards zero), knockout (flux goes to zero) and flux reversal (sign changes).

In [ ]:
from cameo.strain_design.deterministic import DifferentialFVA

Set up a reference model that represents the wild-type state: anaerobic growth at the maximum growth rate. We first close the oxygen exchange on `model` (which becomes the *design space* model), then copy it and fix the reference's growth rate at its optimum (as in cameo's own example).

In [ ]:
model.reactions.EX_o2_e.lower_bound = 0
reference_model = model.copy()
biomass_rxn = reference_model.reactions.BIOMASS_Ecoli_core_w_GAM
biomass_rxn.lower_bound = reference_model.slim_optimize()  # reference: maximal (wild-type) growth
target = reference_model.metabolites.succ_e

Set up the differential flux variability analysis. `variables=[biomass_rxn]` means the grid is laid out along the growth axis of the production envelope, `points=10` gives 10 grid points, and `normalize_ranges_by=biomass_rxn` divides all flux ranges by the growth rate, so that states with very different growth rates can be compared per unit of biomass.

In [ ]:
diff_fva = DifferentialFVA(design_space_model=model,
                           reference_model=reference_model,
                           objective=target,
                           variables=[biomass_rxn],
                           normalize_ranges_by=biomass_rxn,
                           points=10)

Run differential flux variability analysis (only on the surface of the production envelope, i.e. the points with maximal succinate for each growth rate). The plot shows the envelope with the grid points (red) at which FVA was computed.

In [ ]:
%time result = diff_fva.run(surface_only=True)
result.plot(plotter)

Compare the flux ranges of a few reactions between the reference (WT) and grid point 5, a state with intermediate growth and succinate production.

In [ ]:
result.plot(plotter, 5, variables=['FBP', 'G6PDH2r', 'PGL', 'PGK'])

The result object is interactive (in Jupyter, move the slider to choose a grid point). For each reaction it reports the reference and design flux ranges, the gap between them normalised by growth (`normalized_gaps`), and flags for `KO`, `flux_reversal` and `suddenly_essential`. Large positive normalised gaps are over-expression candidates, large negative gaps down-regulation candidates.

In [ ]:
result

Display grid point 5 on the Escher map of central metabolism. The colours encode the normalised gap of each reaction.

In [ ]:
result.display_on_map(5, map_name="e_coli_core.Core metabolism")

## Exercise (15 min)

1. Use FSEOF to find amplification targets for **ethanol** (`EX_etoh_e`) under anaerobic conditions (the model is still anaerobic from the differential FVA section).
2. Rank reactions by how much their *absolute* flux increases between the lowest and highest enforced ethanol flux. Which of the top reactions are realistic genetic targets, and which are not?

### Solution

Reactions can carry negative flux when they run in the direction opposite to how they are written, so we compare absolute fluxes.

In [ ]:
fseof_etoh = FSEOF(model)
result_etoh = fseof_etoh.run(target=model.reactions.EX_etoh_e)
fluxes = result_etoh.data_frame.abs()
increase = (fluxes.iloc[:, -1] - fluxes.iloc[:, 0]).sort_values(ascending=False)
increase.round(2)

The largest increases are CO₂ transport (CO2t) and pyruvate dehydrogenase (PDH), followed by the ethanol pathway itself (ACALD, ALCD2x and ethanol transport ETOHt2r), and the ATP maintenance reaction (ATPM).

* **Not targets:** exchange and transport reactions (EX_etoh_e, ETOHt2r, CO2t) simply follow the product; ATPM is a modelling pseudo-reaction whose flux rises because excess ATP is no longer used for growth.
* **Realistic targets:** the ethanol pathway enzymes (ACALD and ALCD2x, both catalysed by AdhE) and **PDH**. PDH produces acetyl-CoA plus an extra NADH per pyruvate (instead of formate via PFL), and that extra NADH is what allows two ethanol per glucose. PDH is normally inactive under anaerobic conditions *in vivo*, which is exactly why this prediction is interesting. A native *E. coli* mutant whose pyruvate dehydrogenase stays active anaerobically does ferment glucose and xylose to ethanol ([Kim et al., 2007](https://doi.org/10.1128/aem.02456-06)).
* Small increases in glycolytic enzymes (PYK, ENO, GAPD) reflect the slightly higher glucose flux per unit of growth.

## References

- Cardoso, Jensen, Lieven et al. (2018). Cameo: A Python Library for Computer Aided Metabolic Engineering and Optimization of Cell Factories. *ACS Synthetic Biology* 7:1163–1166. [doi.org/10.1021/acssynbio.7b00423](https://doi.org/10.1021/acssynbio.7b00423)
- Choi, Lee, Kim et al. (2010). *In Silico* Identification of Gene Amplification Targets for Improvement of Lycopene Production. *Applied and Environmental Microbiology* 76:3097–3105. [doi.org/10.1128/aem.00115-10](https://doi.org/10.1128/aem.00115-10)
- Kim, Ingram, Shanmugam (2007). Construction of an *Escherichia coli* K-12 Mutant for Homoethanologenic Fermentation of Glucose or Xylose without Foreign Genes. *Applied and Environmental Microbiology* 73:1766–1771. [doi.org/10.1128/aem.02456-06](https://doi.org/10.1128/aem.02456-06)